# 3DGS Paper Vector Database

Reads `annotated-abstracts.csv`, embeds each rhetorical-move section using a
sentence-transformer model, and stores everything in a `sqlite-vec` database.

**Schema**
- `papers` — metadata table: `id`, `filename`, + 8 section text columns
- `vec_full` — one embedding per paper (full abstract concatenated); for general paper discovery
- `vec_sections` — one embedding per (paper × section); `section` is a partition key so you can search *within* a specific rhetorical move

**Downstream queries this enables**
- *"Find papers similar to X"* → KNN on `vec_full`
- *"Find papers whose motivation is about real-time rendering"* → KNN on `vec_sections` filtered by `section = 'motivation'`
- *"Find papers with a contribution similar to this method"* → KNN on `vec_sections` filtered by `section = 'contribution'`

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q sqlite-vec sentence-transformers pandas

In [ ]:
# ── Upload CSV ────────────────────────────────────────────────────────────────
# Run this cell to upload annotated-abstracts.csv from your machine.
# Skip and set CSV_PATH manually if running outside Colab.

try:
    from google.colab import files
    uploaded = files.upload()          # select annotated-abstracts.csv
    CSV_PATH = list(uploaded.keys())[0]
    print(f'Uploaded: {CSV_PATH}')
except ImportError:
    CSV_PATH = 'annotated-abstracts.csv'   # local fallback
    print(f'Not in Colab — using: {CSV_PATH}')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import pandas as pd

# Embedding model — 384-dim, fast on CPU, strong semantic quality
# Swap for 'BAAI/bge-base-en-v1.5' (768-dim) for higher quality at the cost of speed
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
DB_PATH     = '3dgs_papers.db'

SECTION_COLS = [
    'topic',
    'motivation',
    'contribution',
    'detail_nuance',
    'evidence_contribution_2',
    'weaker_result',
    'narrow_impact',
    'broad_impact',
]

df = pd.read_csv(CSV_PATH).fillna('')
print(f'Loaded {len(df)} papers')
df.head(2)

In [ ]:
# ── Generate embeddings ───────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np

print(f'Loading model: {EMBED_MODEL} ...')
model = SentenceTransformer(EMBED_MODEL)
DIM   = model.get_sentence_embedding_dimension()
print(f'Embedding dimension: {DIM}')

def embed_batch(texts: list[str]) -> np.ndarray:
    """Embed a list of strings; replace empty strings with zero vectors."""
    if not texts:
        return np.zeros((0, DIM), dtype=np.float32)
    filled  = [t if t.strip() else '[empty]' for t in texts]
    vecs    = model.encode(filled, show_progress_bar=True, batch_size=64,
                           normalize_embeddings=True)
    # Zero out vectors that were originally empty
    for i, t in enumerate(texts):
        if not t.strip():
            vecs[i] = 0.0
    return vecs.astype(np.float32)

# Full abstract = all 8 sections concatenated (weighted toward Topic + Contribution)
print('\nEmbedding full abstracts ...')
full_texts = df[SECTION_COLS].apply(
    lambda row: ' '.join(row.values), axis=1
).tolist()
full_vecs = embed_batch(full_texts)   # shape: (N, DIM)

# Per-section embeddings
section_vecs = {}   # col -> np.ndarray shape (N, DIM)
for col in SECTION_COLS:
    print(f'Embedding {col} ...')
    section_vecs[col] = embed_batch(df[col].tolist())

print('\nAll embeddings generated.')

In [ ]:
# ── Build sqlite-vec database ─────────────────────────────────────────────────
import sqlite3
import sqlite_vec
import struct
import os

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

con = sqlite3.connect(DB_PATH)
con.enable_load_extension(True)
sqlite_vec.load(con)
con.enable_load_extension(False)

# Verify extension loaded
vec_version, = con.execute('SELECT vec_version()').fetchone()
print(f'sqlite-vec version: {vec_version}')

def to_blob(vec: np.ndarray) -> bytes:
    """Pack a float32 numpy vector into a sqlite-vec binary blob."""
    return struct.pack(f'{len(vec)}f', *vec)

# ── papers (metadata) ─────────────────────────────────────────────────────────
con.execute('DROP TABLE IF EXISTS papers')
cols_ddl = ', '.join(f'{c} TEXT' for c in SECTION_COLS)
con.execute(f'''
    CREATE TABLE papers (
        id       INTEGER PRIMARY KEY AUTOINCREMENT,
        filename TEXT NOT NULL,
        {cols_ddl}
    )
''')

col_list = ', '.join(['filename'] + SECTION_COLS)
placeholders = ', '.join(['?'] * (1 + len(SECTION_COLS)))
con.executemany(
    f'INSERT INTO papers ({col_list}) VALUES ({placeholders})',
    [
        tuple([row['filename']] + [row[c] for c in SECTION_COLS])
        for _, row in df.iterrows()
    ]
)
print(f'Inserted {len(df)} rows into papers')

# ── vec_full (one embedding per paper) ────────────────────────────────────────
con.execute('DROP TABLE IF EXISTS vec_full')
con.execute(f'''
    CREATE VIRTUAL TABLE vec_full USING vec0(
        paper_id INTEGER PRIMARY KEY,
        embedding FLOAT[{DIM}]
    )
''')

con.executemany(
    'INSERT INTO vec_full (paper_id, embedding) VALUES (?, ?)',
    [
        (int(i + 1), to_blob(full_vecs[i]))
        for i in range(len(df))
    ]
)
print(f'Inserted {len(df)} rows into vec_full')

# ── vec_sections (one embedding per paper × section) ──────────────────────────
# section is a partition key — lets you constrain KNN to a single rhetorical move
con.execute('DROP TABLE IF EXISTS vec_sections')
con.execute(f'''
    CREATE VIRTUAL TABLE vec_sections USING vec0(
        paper_id  INTEGER,
        section   TEXT PARTITION KEY,
        embedding FLOAT[{DIM}]
    )
''')

section_rows = []
for col in SECTION_COLS:
    for i in range(len(df)):
        section_rows.append((
            int(i + 1),
            col,
            to_blob(section_vecs[col][i])
        ))

con.executemany(
    'INSERT INTO vec_sections (paper_id, section, embedding) VALUES (?, ?, ?)',
    section_rows
)
print(f'Inserted {len(section_rows)} rows into vec_sections ({len(SECTION_COLS)} sections × {len(df)} papers)')

con.commit()
print(f'\nDatabase saved to: {DB_PATH} ({os.path.getsize(DB_PATH) / 1024 / 1024:.1f} MB)')

In [ ]:
# ── Demo: semantic paper search (vec_full) ────────────────────────────────────

def search_papers(query: str, k: int = 5):
    """Find the k most similar papers to a free-text query."""
    q_vec = model.encode([query], normalize_embeddings=True).astype(np.float32)
    results = con.execute('''
        SELECT p.filename, p.topic, vf.distance
        FROM vec_full vf
        JOIN papers p ON p.id = vf.paper_id
        WHERE vf.embedding MATCH ?
          AND k = ?
        ORDER BY vf.distance
    ''', [to_blob(q_vec[0]), k]).fetchall()

    print(f'Query: "{query}"\n')
    for filename, topic, dist in results:
        print(f'  [{dist:.4f}] {filename}')
        print(f'           {topic[:120]}\n')

search_papers('efficient real-time rendering of large outdoor scenes')

In [ ]:
# ── Demo: search within a specific rhetorical move (vec_sections) ─────────────

def search_section(query: str, section: str, k: int = 5):
    """
    Find papers whose <section> text is most similar to the query.
    section: one of topic | motivation | contribution | detail_nuance |
             evidence_contribution_2 | weaker_result | narrow_impact | broad_impact
    """
    assert section in SECTION_COLS, f'Unknown section: {section!r}. Choose from: {SECTION_COLS}'
    q_vec = model.encode([query], normalize_embeddings=True).astype(np.float32)
    # Column name must be interpolated in Python — SQLite params cannot bind identifiers
    results = con.execute(f'''
        SELECT p.filename, p.{section}, vs.distance
        FROM vec_sections vs
        JOIN papers p ON p.id = vs.paper_id
        WHERE vs.embedding MATCH ?
          AND vs.section = ?
          AND k = ?
        ORDER BY vs.distance
    ''', [to_blob(q_vec[0]), section, k]).fetchall()

    print(f'Section: {section!r}  |  Query: "{query}"\n')
    for filename, text, dist in results:
        print(f'  [{dist:.4f}] {filename}')
        print(f'           {text[:120]}\n')

# Find papers whose *motivation* is about the lack of dynamic scene support
search_section(
    'existing methods cannot handle dynamic or deformable objects',
    section='motivation'
)

# Find papers whose *contribution* involves compression
search_section(
    'compress 3D Gaussian representations to reduce storage size',
    section='contribution'
)

In [ ]:
# ── Demo: find papers similar to a specific paper ─────────────────────────────

def similar_to(filename: str, k: int = 5):
    """Find papers most similar to the given paper (by filename slug)."""
    row = con.execute(
        'SELECT id FROM papers WHERE filename = ?', [filename]
    ).fetchone()
    if not row:
        # Partial match
        row = con.execute(
            "SELECT id FROM papers WHERE filename LIKE ?", [f'%{filename}%']
        ).fetchone()
    if not row:
        print(f'Paper not found: {filename}')
        return

    paper_id = row[0]
    ref_vec_blob, = con.execute(
        'SELECT embedding FROM vec_full WHERE paper_id = ?', [paper_id]
    ).fetchone()

    results = con.execute('''
        SELECT p.filename, p.topic, vf.distance
        FROM vec_full vf
        JOIN papers p ON p.id = vf.paper_id
        WHERE vf.embedding MATCH ?
          AND k = ?
          AND vf.paper_id != ?
        ORDER BY vf.distance
    ''', [ref_vec_blob, k, paper_id]).fetchall()

    print(f'Papers most similar to: {filename}\n')
    for fname, topic, dist in results:
        print(f'  [{dist:.4f}] {fname}')
        print(f'           {topic[:120]}\n')

# Try with the first filename in the dataset
first_filename = con.execute('SELECT filename FROM papers LIMIT 1').fetchone()[0]
similar_to(first_filename)

In [ ]:
# ── Download the database ─────────────────────────────────────────────────────
try:
    from google.colab import files
    files.download(DB_PATH)
    print(f'Downloading {DB_PATH}')
except ImportError:
    print(f'Database at: {os.path.abspath(DB_PATH)}')